### Importing Libraries

In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from math import sqrt
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

### Loading Down Stairs Data (dws_1)

In [7]:
DATA_DIR = Path("A_DeviceMotion_data")
ACTIVITY = "dws_1"   # walking down stairs, trial 1
FS = 50              # DeviceMotion sampling rate (Hz)

# One CSV per subject: sub_1.csv ... sub_24.csv
files = sorted(
    (DATA_DIR / ACTIVITY).glob("sub_*.csv"),
    key=lambda p: int(p.stem.split("_")[1]),
)

dws = {}
for f in files:
    df = pd.read_csv(f, index_col=0)          # first column is the sample counter
    df.index.name = "sample"
    df.insert(0, "time_s", df.index / FS)     # keep RangeIndex for statsmodels, add real time
    dws[int(f.stem.split("_")[1])] = df

# Long format across all subjects, for grouped analysis later
dws_all = pd.concat(dws, names=["subject", "sample"]).reset_index("subject")

print("Activity:", ACTIVITY)
print("Subjects:", len(dws), "->", sorted(dws))
print("Total samples:", len(dws_all), f"({len(dws_all) / FS:.1f} s at {FS} Hz)")
print("\nSamples per subject:")
print(dws_all.groupby("subject").size().describe()[["min", "mean", "max"]])

print("\nColumns:", list(dws_all.columns))
print("\nMissing values:")
print(dws_all.isna().sum())

# Single subject used as the working series for the univariate models
sub = dws[1]
print("\nSubject 1 shape:", sub.shape, "| duration:", f"{sub['time_s'].iloc[-1]:.2f} s")
display(sub.head())
display(sub.describe().T)


Activity: dws_1
Subjects: 24 -> [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
Total samples: 50246 (1004.9 s at 50 Hz)

Samples per subject:
min     1522.000000
mean    2093.583333
max     2946.000000
dtype: float64

Columns: ['subject', 'time_s', 'attitude.roll', 'attitude.pitch', 'attitude.yaw', 'gravity.x', 'gravity.y', 'gravity.z', 'rotationRate.x', 'rotationRate.y', 'rotationRate.z', 'userAcceleration.x', 'userAcceleration.y', 'userAcceleration.z']

Missing values:
subject               0
time_s                0
attitude.roll         0
attitude.pitch        0
attitude.yaw          0
gravity.x             0
gravity.y             0
gravity.z             0
rotationRate.x        0
rotationRate.y        0
rotationRate.z        0
userAcceleration.x    0
userAcceleration.y    0
userAcceleration.z    0
dtype: int64

Subject 1 shape: (1751, 13) | duration: 35.00 s


,time_s,attitude.roll,attitude.pitch,attitude.yaw,gravity.x,gravity.y,gravity.z,rotationRate.x,rotationRate.y,rotationRate.z,userAcceleration.x,userAcceleration.y,userAcceleration.z
sample,,,,,,,,,,,,,
0,0.00,1.528132,-0.733896,0.696372,0.741895,0.669768,-0.031672,0.316738,0.778180,1.082764,0.294894,-0.184493,0.377542
1,0.02,1.527992,-0.716987,0.677762,0.753099,0.657116,-0.032255,0.842032,0.424446,0.643574,0.219405,0.035846,0.114866
2,0.04,1.527765,-0.706999,0.670951,0.759611,0.649555,-0.032707,-0.138143,-0.040741,0.343563,0.010714,0.134701,-0.167808
3,0.06,1.516768,-0.704678,0.675735,0.760709,0.647788,-0.041140,-0.025005,-1.048717,0.035860,-0.008389,0.136788,0.094958
4,0.08,1.493941,-0.703918,0.672994,0.760062,0.647210,-0.058530,0.114253,-0.912890,0.047341,0.199441,0.353996,-0.044299


,count,mean,std,min,25%,50%,75%,max
time_s,1751.0,17.500000,10.112290,0.000000,8.750000,17.500000,26.250000,35.000000
attitude.roll,1751.0,1.387143,0.242061,0.794846,1.197726,1.432454,1.559388,1.991564
attitude.pitch,1751.0,-0.658759,0.154975,-1.028001,-0.776018,-0.655018,-0.528868,-0.353937
attitude.yaw,1751.0,-0.645835,1.885691,-3.131776,-2.468795,-0.650467,0.820519,3.137648
gravity.x,1751.0,0.745272,0.097532,0.500776,0.681659,0.748291,0.826994,0.911532
gravity.y,1751.0,0.604755,0.120785,0.346594,0.504556,0.609173,0.700443,0.856268
gravity.z,1751.0,-0.141934,0.186086,-0.552549,-0.294859,-0.103107,-0.008844,0.293529
rotationRate.x,1751.0,-0.313635,1.079116,-3.899313,-1.065276,-0.298466,0.475518,3.391939
rotationRate.y,1751.0,-0.245480,1.964371,-6.689527,-1.446409,-0.150008,0.956249,6.040657
rotationRate.z,1751.0,0.141903,0.702076,-2.182188,-0.306418,0.096523,0.567980,2.905782
